In [1]:
import pandas as pd
import requests as req
import geopandas as gpd

# Load crimes
df = pd.read_csv('../notebooks/data/raw/Crimes_-_One_year_prior_to_present.csv')

# Load communities GeoJSON
communities = gpd.read_file("https://data.cityofchicago.org/resource/igwz-8jzy.geojson")
# Parse coordinates
df[['lat', 'lon']] = (
    df['LOCATION']
        .str.strip('()')
        .str.split(',', expand=True)
        .astype(float)
)

# Create geometry
df['points'] = gpd.points_from_xy(df['lon'], df['lat'])

# Convert to GeoDataFrame
df = gpd.GeoDataFrame(
    df,
    geometry='points',
    crs="EPSG:4326"
)

c_df = gpd.sjoin(df, communities, predicate="within")

socieo = pd.read_csv('../notebooks/data/raw/Census_Data_-_Selected_socioeconomic_indicators_in_Chicago,_2008_–_2012.csv')
education = pd.read_csv('../notebooks/data/raw/Chicago_Public_Schools_-_Progress_Report_Cards_(2011-2012).csv')

def clean_name(s):
    return (
        s.str.lower()
         .str.strip()
         .str.replace(r"[^\w\s]", "", regex=True)
         .str.replace(r"\s+", " ", regex=True)
    )

c_df["community_clean"] = clean_name(c_df["community"])
socieo["community_clean"] = clean_name(socieo["COMMUNITY AREA NAME"])
education["community_clean"] = clean_name(education["Community Area Name"])


merged = pd.merge(
    c_df,
    socieo,
    on="community_clean"
)

merged = pd.merge(
    merged,
    education,
    on="community_clean"
)

merged


,CASE#,DATE OF OCCURRENCE,BLOCK,IUCR,PRIMARY DESCRIPTION,SECONDARY DESCRIPTION,LOCATION DESCRIPTION,ARREST,DOMESTIC,BEAT,...,RCDTS Code,X_COORDINATE,Y_COORDINATE,Latitude,Longitude,Community Area Number_y,Community Area Name,Ward,Police District,Location
0,JJ518420,01/11/2025 05:10:00 AM,050XX W CONGRESS PKWY,0486,BATTERY,DOMESTIC BATTERY SIMPLE,APARTMENT,N,Y,1533,...,150000000000000,1139494.763,1901274.258,41.885205,-87.763212,25,AUSTIN,28,15,"(41.88520477, -87.76321191)"
1,JJ518420,01/11/2025 05:10:00 AM,050XX W CONGRESS PKWY,0486,BATTERY,DOMESTIC BATTERY SIMPLE,APARTMENT,N,Y,1533,...,150000000000000,1139494.763,1901274.258,41.885205,-87.763212,25,AUSTIN,28,15,"(41.88520477, -87.76321191)"
2,JJ518420,01/11/2025 05:10:00 AM,050XX W CONGRESS PKWY,0486,BATTERY,DOMESTIC BATTERY SIMPLE,APARTMENT,N,Y,1533,...,150000000000000,1138584.642,1901378.002,41.885506,-87.766552,25,AUSTIN,29,15,"(41.885506, -87.76655157)"
3,JJ518420,01/11/2025 05:10:00 AM,050XX W CONGRESS PKWY,0486,BATTERY,DOMESTIC BATTERY SIMPLE,APARTMENT,N,Y,1533,...,150000000000000,1138362.536,1909130.405,41.906784,-87.767179,25,AUSTIN,29,25,"(41.90678358, -87.76717934)"
4,JJ518420,01/11/2025 05:10:00 AM,050XX W CONGRESS PKWY,0486,BATTERY,DOMESTIC BATTERY SIMPLE,APARTMENT,N,Y,1533,...,150000000000000,1138633.137,1902956.198,41.889836,-87.766335,25,AUSTIN,29,15,"(41.88983589, -87.76633519)"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2284586,JK111298,01/10/2026 12:00:00 AM,035XX N CLARK ST,0870,THEFT,POCKET-PICKING,BAR OR TAVERN,N,N,1924,...,150000000000000,1164975.217,1926813.581,41.954784,-87.668916,6,LAKE VIEW,47,19,"(41.95478361, -87.66891643)"
2284587,JK111298,01/10/2026 12:00:00 AM,035XX N CLARK ST,0870,THEFT,POCKET-PICKING,BAR OR TAVERN,N,N,1924,...,150000000000000,1168495.925,1919310.804,41.934120,-87.656192,6,LAKE VIEW,44,19,"(41.93412004, -87.65619167)"
2284588,JK111298,01/10/2026 12:00:00 AM,035XX N CLARK ST,0870,THEFT,POCKET-PICKING,BAR OR TAVERN,N,N,1924,...,150000000000000,1171628.585,1922072.580,41.941630,-87.644598,6,LAKE VIEW,44,19,"(41.94162999, -87.64459776)"
2284589,JK111298,01/10/2026 12:00:00 AM,035XX N CLARK ST,0870,THEFT,POCKET-PICKING,BAR OR TAVERN,N,N,1924,...,150000000000000,1164156.393,1927985.538,41.958017,-87.671893,6,LAKE VIEW,47,19,"(41.9580169, -87.6718933)"


In [2]:
c_df.columns

Index(['CASE#', 'DATE  OF OCCURRENCE', 'BLOCK', ' IUCR',
       ' PRIMARY DESCRIPTION', ' SECONDARY DESCRIPTION',
       ' LOCATION DESCRIPTION', 'ARREST', 'DOMESTIC', 'BEAT', 'WARD', 'FBI CD',
       'X COORDINATE', 'Y COORDINATE', 'LATITUDE', 'LONGITUDE', 'LOCATION',
       'lat', 'lon', 'points', 'index_right', 'community', 'shape_area',
       'area_num_1', 'area_numbe', 'shape_len', 'community_clean'],
      dtype='object')

In [3]:
c_df.groupby(['community',' PRIMARY DESCRIPTION']).size().sort_values(ascending=False)

community         PRIMARY DESCRIPTION            
NEAR NORTH SIDE  THEFT                               4659
LOOP             THEFT                               3321
NEAR WEST SIDE   THEFT                               3015
AUSTIN           BATTERY                             2572
LAKE VIEW        THEFT                               2332
                                                     ... 
EDGEWATER        PROSTITUTION                           1
EDISON PARK      ARSON                                  1
                 INTERFERENCE WITH PUBLIC OFFICER       1
                 KIDNAPPING                             1
FOREST GLEN      INTIMIDATION                           1
Length: 1807, dtype: int64

In [4]:
c_df.groupby([' PRIMARY DESCRIPTION']).size().sort_values(ascending=False)

 PRIMARY DESCRIPTION
THEFT                                54166
BATTERY                              42284
CRIMINAL DAMAGE                      26155
ASSAULT                              21372
MOTOR VEHICLE THEFT                  17176
OTHER OFFENSE                        16360
DECEPTIVE PRACTICE                   14034
BURGLARY                              9665
NARCOTICS                             7357
ROBBERY                               5709
WEAPONS VIOLATION                     5381
CRIMINAL TRESPASS                     5314
CRIMINAL SEXUAL ASSAULT               1623
OFFENSE INVOLVING CHILDREN            1513
SEX OFFENSE                           1268
PUBLIC PEACE VIOLATION                1056
INTERFERENCE WITH PUBLIC OFFICER       917
STALKING                               558
HOMICIDE                               422
ARSON                                  362
CONCEALED CARRY LICENSE VIOLATION      266
LIQUOR LAW VIOLATION                   191
PROSTITUTION                     

In [5]:
pd.read_csv('../notebooks/data/raw/Chicago_Public_Schools_-_Progress_Report_Cards_(2011-2012).csv')

,School ID,Name of School,"Elementary, Middle, or High School",Street Address,City,State,ZIP Code,Phone Number,Link,Network Manager,...,RCDTS Code,X_COORDINATE,Y_COORDINATE,Latitude,Longitude,Community Area Number,Community Area Name,Ward,Police District,Location
0,610038,Abraham Lincoln Elementary School,ES,615 W Kemper Pl,Chicago,IL,60614,(773) 534-5720,http://schoolreports.cps.edu/SchoolProgressRep...,Fullerton Elementary Network,...,150000000000000,1171699.458,1915829.428,41.924497,-87.644522,7,LINCOLN PARK,43,18,"(41.92449696, -87.64452163)"
1,610281,Adam Clayton Powell Paideia Community Academy ...,ES,7511 S South Shore Dr,Chicago,IL,60649,(773) 535-6650,http://schoolreports.cps.edu/SchoolProgressRep...,Skyway Elementary Network,...,150000000000000,1196129.985,1856209.466,41.760324,-87.556736,43,SOUTH SHORE,7,4,"(41.76032435, -87.55673627)"
2,610185,Adlai E Stevenson Elementary School,ES,8010 S Kostner Ave,Chicago,IL,60652,(773) 535-2280,http://schoolreports.cps.edu/SchoolProgressRep...,Midway Elementary Network,...,150000000000000,1148427.165,1851012.215,41.747111,-87.731702,70,ASHBURN,13,8,"(41.74711093, -87.73170248)"
3,609993,Agustin Lara Elementary Academy,ES,4619 S Wolcott Ave,Chicago,IL,60609,(773) 535-4389,http://schoolreports.cps.edu/SchoolProgressRep...,Pershing Elementary Network,...,150000000000000,1164504.290,1873959.199,41.809757,-87.672145,61,NEW CITY,20,9,"(41.8097569, -87.6721446)"
4,610513,Air Force Academy High School,HS,3630 S Wells St,Chicago,IL,60609,(773) 535-1590,http://schoolreports.cps.edu/SchoolProgressRep...,Southwest Side High School Network,...,150000000000000,1175177.622,1880745.126,41.828146,-87.632794,34,ARMOUR SQUARE,11,9,"(41.82814609, -87.63279369)"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
561,609844,William W Carter Elementary School,ES,5740 S Michigan Ave,Chicago,IL,60637,(773) 535-0860,http://schoolreports.cps.edu/SchoolProgressRep...,Burnham Park Elementary Network,...,150000000000000,1178101.365,1866810.123,41.789841,-87.622490,40,WASHINGTON PARK,20,2,"(41.78984129, -87.62248974)"
562,610308,Wilma Rudolph Elementary Learning Center,ES,110 N Paulina St,Chicago,IL,60612,(773) 534-7460,http://schoolreports.cps.edu/SchoolProgressRep...,Fulton Elementary Network,...,150000000000000,1165013.632,1900863.727,41.883575,-87.669514,28,NEAR WEST SIDE,27,13,"(41.88357478, -87.66951363)"
563,610088,Wolfgang A Mozart Elementary School,ES,2200 N Hamlin Ave,Chicago,IL,60647,(773) 534-4160,http://schoolreports.cps.edu/SchoolProgressRep...,Fullerton Elementary Network,...,150000000000000,1150644.396,1914368.955,41.920927,-87.721925,22,LOGAN SQUARE,35,25,"(41.92092734, -87.72192541)"
564,609977,Woodlawn Community Elementary School,ES,6657 S Kimbark Ave,Chicago,IL,60637,(773) 535-0801,http://schoolreports.cps.edu/SchoolProgressRep...,Burnham Park Elementary Network,...,150000000000000,1185825.188,1860883.579,41.773400,-87.594356,42,WOODLAWN,5,3,"(41.77339962, -87.59435584)"


In [6]:
pd.read_csv('../notebooks/data/raw/Census_Data_-_Selected_socioeconomic_indicators_in_Chicago,_2008_–_2012.csv')

,Community Area Number,COMMUNITY AREA NAME,PERCENT OF HOUSING CROWDED,PERCENT HOUSEHOLDS BELOW POVERTY,PERCENT AGED 16+ UNEMPLOYED,PERCENT AGED 25+ WITHOUT HIGH SCHOOL DIPLOMA,PERCENT AGED UNDER 18 OR OVER 64,PER CAPITA INCOME,HARDSHIP INDEX
0,1.0,Rogers Park,7.7,23.6,8.7,18.2,27.5,23939,39.0
1,2.0,West Ridge,7.8,17.2,8.8,20.8,38.5,23040,46.0
2,3.0,Uptown,3.8,24.0,8.9,11.8,22.2,35787,20.0
3,4.0,Lincoln Square,3.4,10.9,8.2,13.4,25.5,37524,17.0
4,5.0,North Center,0.3,7.5,5.2,4.5,26.2,57123,6.0
...,...,...,...,...,...,...,...,...,...
73,74.0,Mount Greenwood,1.0,3.4,8.7,4.3,36.8,34381,16.0
74,75.0,Morgan Park,0.8,13.2,15.0,10.8,40.3,27149,30.0
75,76.0,O'Hare,3.6,15.4,7.1,10.9,30.3,25828,24.0
76,77.0,Edgewater,4.1,18.2,9.2,9.7,23.8,33385,19.0


,geometry,community,shape_area,area_num_1,area_numbe,shape_len
0,"MULTIPOLYGON (((-87.65456 41.99817, -87.65574 ...",ROGERS PARK,51259902.4506,1,1,34052.3975757
1,"MULTIPOLYGON (((-87.68465 42.01948, -87.68464 ...",WEST RIDGE,98429094.8621,2,2,43020.6894583
2,"MULTIPOLYGON (((-87.64102 41.9548, -87.644 41....",UPTOWN,65095642.7289,3,3,46972.7945549
3,"MULTIPOLYGON (((-87.67441 41.9761, -87.6744 41...",LINCOLN SQUARE,71352328.2399,4,4,36624.6030848
4,"MULTIPOLYGON (((-87.67336 41.93234, -87.67342 ...",NORTH CENTER,57054167.85,5,5,31391.6697542
...,...,...,...,...,...,...
72,"MULTIPOLYGON (((-87.63373 41.72885, -87.63369 ...",WASHINGTON HEIGHTS,79635752.8769,73,73,42222.598163
73,"MULTIPOLYGON (((-87.69646 41.70714, -87.69644 ...",MOUNT GREENWOOD,75584290.0209,74,74,48665.1305392
74,"MULTIPOLYGON (((-87.64215 41.68508, -87.64249 ...",MORGAN PARK,91877340.6988,75,75,46396.419362
75,"MULTIPOLYGON (((-87.83658 41.9864, -87.83658 4...",OHARE,371835607.687,76,76,173625.98466


In [84]:
pd.read_csv('../notebooks/data/raw/Crimes_-_One_year_prior_to_present.csv')

,CASE#,DATE OF OCCURRENCE,BLOCK,IUCR,PRIMARY DESCRIPTION,SECONDARY DESCRIPTION,LOCATION DESCRIPTION,ARREST,DOMESTIC,BEAT,WARD,FBI CD,X COORDINATE,Y COORDINATE,LATITUDE,LONGITUDE,LOCATION
0,JJ518420,01/11/2025 05:10:00 AM,050XX W CONGRESS PKWY,0486,BATTERY,DOMESTIC BATTERY SIMPLE,APARTMENT,N,Y,1533,29.0,08B,1142740.0,1897209.0,41.873989,-87.751396,"(41.873989415, -87.751395901)"
1,JJ111236,01/11/2025 05:30:00 AM,079XX S ESSEX AVE,0610,BURGLARY,FORCIBLE ENTRY,APARTMENT,N,N,422,7.0,05,1194235.0,1852791.0,41.750991,-87.563793,"(41.750990548, -87.563793363)"
2,JJ118671,01/11/2025 05:45:00 AM,003XX W DIVERSEY PKWY,0710,THEFT,THEFT FROM MOTOR VEHICLE,RESIDENCE - GARAGE,N,N,1934,44.0,06,1173363.0,1918916.0,41.932930,-87.638317,"(41.932929806, -87.638317188)"
3,JJ111244,01/11/2025 05:50:00 AM,098XX S EWING AVE,0520,ASSAULT,AGGRAVATED - KNIFE / CUTTING INSTRUMENT,APARTMENT,Y,Y,432,10.0,04A,1202130.0,1840516.0,41.717110,-87.535280,"(41.717109688, -87.535279997)"
4,JJ111231,01/11/2025 05:50:00 AM,071XX S JEFFERY BLVD,1310,CRIMINAL DAMAGE,TO PROPERTY,COMMERCIAL / BUSINESS OFFICE,N,N,333,5.0,14,1190779.0,1858055.0,41.765520,-87.576288,"(41.765519532, -87.576287871)"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
234514,JK109928,01/10/2026 12:00:00 AM,091XX S STONY ISLAND AVE,0560,ASSAULT,SIMPLE,GAS STATION,N,N,413,8.0,08A,1188515.0,1844485.0,41.728337,-87.585019,"(41.728336543, -87.58501858)"
234515,JK110617,01/10/2026 12:00:00 AM,042XX W 76TH ST,0486,BATTERY,DOMESTIC BATTERY SIMPLE,APARTMENT,Y,Y,833,18.0,08B,1148831.0,1853893.0,41.755009,-87.730149,"(41.755008548, -87.730148696)"
234516,JK112103,01/10/2026 12:00:00 AM,012XX N SPAULDING AVE,0910,MOTOR VEHICLE THEFT,AUTOMOBILE,STREET,N,N,1422,26.0,07,1154134.0,1908210.0,41.903958,-87.709268,"(41.903957644, -87.709268417)"
234517,JK109983,01/10/2026 12:00:00 AM,069XX S CLYDE AVE,0820,THEFT,$500 AND UNDER,APARTMENT,N,Y,331,5.0,06,1191411.0,1859329.0,41.769000,-87.573930,"(41.7690002, -87.573930201)"


In [7]:
df = pd.read_csv('../notebooks/data/raw/ACS_5_Year_Data_by_Community_Area_-_Most_Recent_Year_20260208.csv')
df

,ACS Year,Community Area,"Under $25,000","$25,000 to $49,999","$50,000 to $74,999","$75,000 to $125,000","$125,000 +",Male 0 to 17,Male 18 to 24,Male 25 to 34,...,White,Black or African American,American Indian or Alaska Native,Asian,Native Hawaiian or Pacific Islander,Other Race,Multiracial,White Not Hispanic or Latino,Hispanic or Latino,Record ID
0,2023,ALBANY PARK,"1,269","1,916","1,801","2,306","3,379","4,799","2,955","4,513",...,"21,496","2,228",759,"7,124",1,"7,888","8,334","16,115","21,108",2023_ALBANY PARK
1,2023,ARCHER HEIGHTS,223,752,441,795,739,"1,927",732,"1,102",...,"6,232",10,108,679,0,"3,705","3,142","2,043","11,097",2023_ARCHER HEIGHTS
2,2023,ARMOUR SQUARE,701,798,370,637,597,"1,300",487,871,...,"2,556","1,487",107,"8,402",61,212,325,"2,226",565,2023_ARMOUR SQUARE
3,2023,ASHBURN,797,"1,351","1,985","3,014","2,735","5,150","1,964","2,881",...,"11,297","18,124",697,436,0,"7,772","4,517","3,774","19,917",2023_ASHBURN
4,2023,AUBURN GRESHAM,"2,541","2,451","1,592","2,202","1,850","5,803","1,836","2,964",...,760,"43,414",119,399,0,993,798,491,"1,577",2023_AUBURN GRESHAM
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72,2023,WEST LAWN,766,"1,224","1,296","1,909","1,777","4,673","1,865","2,004",...,"13,717",929,585,255,171,"10,645","6,149","3,980","27,275",2023_WEST LAWN
73,2023,WEST PULLMAN,"1,380","1,196",850,"1,273","1,158","3,160","1,306","1,515",...,908,"22,426",0,0,0,790,"1,035",289,"1,856",2023_WEST PULLMAN
74,2023,WEST RIDGE,"2,088","3,577","3,031","4,005","5,327","10,747","3,219","5,519",...,"36,943","9,310",681,"16,696",17,"6,958","8,615","31,486","17,531",2023_WEST RIDGE
75,2023,WEST TOWN,"1,051","1,114","1,197","2,506","10,663","5,722","2,592","16,967",...,"61,759","6,039",147,"5,138",21,"4,650","8,700","54,505","17,099",2023_WEST TOWN
